# SI3003 · Evaluación 2 · Parte II (aplicada) — Notebook base

**Profesor:** Jorge Iván Padilla Buriticá · Universidad EAFIT

Este notebook es **una ayuda, no una solución**. Trae la infraestructura (configuración, carga de datos, utilidades) y deja **esqueletos vacíos** para lo que se califica. Puede borrar, mover o cambiar lo que quiera.

| Etiqueta en la celda | Significado |
|---|---|
| `# [PROVISTO]` | Ya funciona. Léala y entiéndala; puede modificarla. |
| `# [SU TURNO]` | Aquí va **su** código o **su** análisis. |

> ⏱ **Tiempo máximo: 60 minutos.** Si una tarea se complica, avance y deje escrito qué intentó.
> 📨 **Entrega:** un único notebook **ejecutado de arriba hacia abajo**, enviado a **jipadillab@eafit.edu.co** con el asunto `[SI3003-Nombre_Apellido_Grupo_Eval02_practica]`, **hasta las 11:00 a. m.** (la celda final le imprime su asunto).

## ✅ Antes del examen: qué debe tener preparado

- [ ] **Al menos un proveedor de chat con clave** guardada en **Colab Secrets** (🔑 en la barra izquierda, con *acceso al notebook* activado): `NVIDIA_API_KEY`, `GROQ_API_KEY` o `HF_TOKEN`. **Pruébela con un *ping*** y anote el **nombre exacto del modelo** que funcionó (los catálogos cambian).
- [ ] **Modelos locales ya descargados** (ejecute la celda *Precalentamiento* en casa): `gpt2`, el modelo de embeddings oficial (ver `embedding_info.json`), `all-MiniLM-L6-v2` y el tokenizador de `tiktoken`.
- [ ] **Acceso a la carpeta de Drive** del examen y la carpeta `datos_examen/` copiada a Colab (ver celda de datos).
- [ ] El **enunciado PDF** de la Parte II abierto en otra pestaña.
- [ ] **No necesita GPU**: todo corre en CPU.
- [ ] Repase las **reglas de compatibilidad R1–R4** del enunciado (abajo, resumidas).

### Reglas de compatibilidad (resumen)
1. **Mismo modelo de embeddings** (y mismos parámetros: `input_type`/prefijos, `dimensions`, normalización) para indexar y consultar.
2. **Misma dimensión ≠ compatible.** Dos modelos de 384 dimensiones producen espacios distintos.
3. El **LLM de generación es independiente** del embedding: puede cambiar de proveedor de chat sin reindexar.
4. Los **parámetros específicos de un proveedor** (`extra_body` de NVIDIA, `logprobs` en Groq) pueden dar error 400 en otro: aíslelos.

| Proveedor | Chat | Embeddings |
|---|---|---|
| NVIDIA NIM | Sí | Sí (con `input_type` en modelos asimétricos) |
| Groq | Sí | **No** |
| Hugging Face (router) | Sí | No por ese endpoint (use `feature_extraction` o un modelo local) |
| Local | Sí (lento) | Sí |

## 0 · Configuración inicial

In [ ]:
# [PROVISTO] Instalación. En Colab tarda ~1-2 min. Si ya lo instaló, omítala.
%pip install -q openai sentence-transformers rank_bm25 faiss-cpu tiktoken transformers pandas matplotlib python-dotenv

In [ ]:
# [PROVISTO] PRECALENTAMIENTO — ejecútela ANTES del examen (en casa) para dejar los modelos en caché.
# Cambie el primer modelo si embedding_info.json indica otro.
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import tiktoken
_ = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
_ = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
_ = AutoTokenizer.from_pretrained("gpt2"); _ = AutoModelForCausalLM.from_pretrained("gpt2")
_ = tiktoken.get_encoding("cl100k_base")
print("✅ Modelos locales listos")

In [ ]:
# [SU TURNO] Sus datos. Todo lo demás se calcula a partir de aquí.
STUDENT_NAME = "Nombre Apellido"     # TODO
STUDENT_ID   = "0000000000"          # TODO: su documento (solo dígitos)
GROUP        = "00"                  # TODO: su grupo

In [ ]:
# [PROVISTO] Semilla, dígito personal, cronómetro y asunto del correo.
import time, random, numpy as np
assert STUDENT_ID.isdigit() and len(STUDENT_ID) >= 4, "Escriba su documento (solo dígitos)"
SEED = int(STUDENT_ID[-4:])     # últimos 4 dígitos
D    = int(STUDENT_ID[-1])      # último dígito (define su subconjunto en A.3)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
T0 = time.time()
def tiempo(): print(f"⏱ {(time.time()-T0)/60:.1f} min transcurridos de 60")
ASUNTO = f"[SI3003-{STUDENT_NAME.strip().replace(' ', '_')}_{GROUP}_Eval02_practica]"
print("SEED =", SEED, "| D =", D); print("Asunto del correo:", ASUNTO)

In [ ]:
# [PROVISTO] Lectura segura de claves: Colab Secrets -> variable de entorno -> (opcional) getpass.
# NUNCA imprima una clave ni la escriba en una celda.
import os, getpass
def get_secret(name, ask=False):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    v = os.getenv(name)
    if not v and ask:
        v = getpass.getpass(f"{name}: ").strip()
    return v or None

CLAVES_DISPONIBLES = {k: bool(get_secret(k)) for k in ["NVIDIA_API_KEY", "GROQ_API_KEY", "HF_TOKEN"]}
print("Claves disponibles (sin mostrarlas):", CLAVES_DISPONIBLES)

In [ ]:
# [PROVISTO] Carpeta de datos. Elija UNA opción.
from pathlib import Path
# Opción A (Colab + Drive): descomente y ajuste la ruta
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/<ruta-a-la-carpeta>/datos_examen')
# Opción B: suba la carpeta a /content/datos_examen
DATA_DIR = Path("datos_examen")

ESPERADOS = ["yolo_results_scratch.csv", "yolo_results_pretrained.csv", "yolo_results_freeze.csv",
             "yolo_test_preds.json", "yolo_test_gt.json", "embedding_info.json",
             "corpus_reglamento.json", "preguntas_eval.json", "corpus_embeddings.npy",
             "question_embeddings.npy", "faithfulness_cases.json"]
def revisar_archivos():
    ok = True
    for f in ESPERADOS:
        existe = (DATA_DIR / f).exists(); ok &= existe
        print("✅" if existe else "❌", f)
    return ok
revisar_archivos()

## SECCIÓN A — CNN y detección con YOLO (33 pts)

### A.1 · Dimensiones y parámetros (5 pts)
Qué debe entregar: `conv_out` implementada · 3 configuraciones validadas con `assert` contra `torch.nn.Conv2d` (**una con stride 2 y una con padding**) · la red de la **Pregunta 1 de la Parte I** construida en PyTorch, con la forma de cada capa y el total de parámetros, comparados con su cálculo a mano.

In [ ]:
# [SU TURNO] A.1
def conv_out(W, K, P, S):
    # TODO: fórmula de clase
    raise NotImplementedError("TODO")

# TODO: tres configuraciones (W, K, P, S) y assert contra torch.nn.Conv2d.
# Pista: y = torch.nn.Conv2d(C_in, C_out, K, stride=S, padding=P)(torch.zeros(1, C_in, W, W)); y.shape[-1]

In [ ]:
# [SU TURNO] A.1 — red de la Pregunta 1 (Parte I). Imprima la forma tras cada capa y el total de parámetros.
# Pista: sum(p.numel() for p in modelo.parameters())

### A.2 · Comparación de las tres estrategias (12 pts)
Qué debe entregar: **(4)** gráfica con pérdida de caja de entrenamiento y mAP50 por época de los 3 modelos · **(3)** tabla con valores **finales** de mAP50, mAP50-95, precision, recall y tiempo total · **(5)** análisis (máx. 150 palabras) con evidencia de *sus* gráficas.

In [ ]:
# [PROVISTO] Carga de los tres CSV. Revise los nombres de columna antes de graficar.
import pandas as pd, matplotlib.pyplot as plt
runs = {}
for nombre in ["scratch", "pretrained", "freeze"]:
    df = pd.read_csv(DATA_DIR / f"yolo_results_{nombre}.csv")
    df.columns = df.columns.str.strip()          # Ultralytics deja espacios en los nombres
    runs[nombre] = df
print(list(runs["pretrained"].columns))
runs["pretrained"].head(3)

In [ ]:
# [SU TURNO] A.2 — gráficas (pérdida de caja y mAP50 por época) y tabla de valores finales.

**[SU TURNO] A.2 — Análisis (máx. 150 palabras).** Explique el orden de desempeño con evidencia de sus gráficas y tabla. ¿Qué estrategia usaría con un dataset 10 veces mayor y de un dominio muy distinto al de preentrenamiento, y por qué? Relacione con la Clase 8.

*(escriba aquí)*

### A.3 · IoU, NMS y evaluación manual (16 pts)
Qué debe entregar: **(5)** `iou` y `nms` con `assert` (incluya un caso de NMS con 4 cajas que elimine un duplicado) · **(3)** NMS **por clase** sobre sus 10 imágenes, con conteo de cajas antes/después · **(6)** TP, FP, FN, precision y recall con `conf=0.25` y `conf=0.5`, en una tabla · **(2)** explicación del *trade-off* con *sus* números.

In [ ]:
# [PROVISTO] Carga de predicciones/ground truth y SU subconjunto de 10 imágenes (image_id % 3 == D % 3).
import json
preds = json.load(open(DATA_DIR / "yolo_test_preds.json"))   # {image_id: [{box, cls, conf}, ...]}
gts   = json.load(open(DATA_DIR / "yolo_test_gt.json"))      # {image_id: [{box, cls}, ...]}
mis_ids = [i for i in sorted(preds, key=int) if int(i) % 3 == D % 3]
assert len(mis_ids) == 10, len(mis_ids)
print("Sus imágenes:", mis_ids)

In [ ]:
# [SU TURNO] A.3 — iou y nms
def iou(a, b):          # cajas (x1, y1, x2, y2)
    raise NotImplementedError("TODO")

def nms(boxes, scores, iou_thr=0.5, score_thr=0.0):   # devuelve los índices que se conservan
    raise NotImplementedError("TODO")

# TODO: asserts. Por ejemplo: iou((2,2,6,6), (4,4,8,8)) debe valer 4/28.
# TODO: caso sintético de NMS con 4 cajas.

In [ ]:
# [SU TURNO] A.3 — NMS por clase sobre sus 10 imágenes (antes/después), emparejamiento
# (IoU > 0.5 y misma clase; cada caja real se empareja una sola vez) y tabla TP/FP/FN/P/R para conf = 0.25 y 0.5.

**[SU TURNO] A.3 — Explicación (2 pts).** ¿Por qué al subir el umbral de confianza sube la precision y baja el recall? Use *sus* números.

*(escriba aquí)*

## SECCIÓN B — Transformers y NLP (30 pts)

### B.1 · Atención causal desde cero (12 pts)
Qué debe entregar: **(6)** `causal_self_attention(X, Wq, Wk, Wv)` de **una cabeza**, sin `nn.MultiheadAttention` ni funciones de atención ya hechas · **(3)** tres `assert`: filas suman 1, triángulo superior = 0, coincide (tol. 1e-5) con `F.scaled_dot_product_attention(..., is_causal=True)` · **(3)** reproduzca el ejemplo de la **Pregunta 4 de la Parte I** y compárelo con su resultado a mano.

In [ ]:
# [PROVISTO] Datos de prueba generados con SU semilla (5 tokens, dimensión 8; W de 8x4).
import torch, math, torch.nn.functional as F
g  = torch.Generator().manual_seed(SEED)
X  = torch.randn(5, 8, generator=g)
Wq = torch.randn(8, 4, generator=g); Wk = torch.randn(8, 4, generator=g); Wv = torch.randn(8, 4, generator=g)

In [ ]:
# [SU TURNO] B.1
def causal_self_attention(X, Wq, Wk, Wv):
    # TODO: devuelva (salida, matriz_de_atención)
    raise NotImplementedError("TODO")

# TODO: tres asserts y el ejemplo de la Pregunta 4 (q3, k1..k3, v1..v3 dados en el enunciado impreso).

### B.2 · Generación autorregresiva con un modelo real (10 pts)
Qué debe entregar: **(3)** los 10 tokens más probables tras *"The dog chased the"*, con su probabilidad · **(4)** para $T\in\{0.2,1,2\}$: probabilidad del top-1 y **entropía** · **(3)** explicación relacionada con la Pregunta 5(b) de la Parte I.
> Esta sección necesita **logits locales**: Groq rechaza `logprobs` y los endpoints de chat no devuelven la distribución completa.

In [ ]:
# [PROVISTO] Modelo local y logits del siguiente token.
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("gpt2")
lm  = AutoModelForCausalLM.from_pretrained("gpt2").eval()
ids = tok("The dog chased the", return_tensors="pt")
with torch.no_grad():
    logits = lm(**ids).logits[0, -1]        # vector de logits sobre todo el vocabulario
print(logits.shape)

In [ ]:
# [SU TURNO] B.2 — top-10 con probabilidades; softmax con temperatura; entropía.

**[SU TURNO] B.2 — Explicación (3 pts).** *(escriba aquí)*

### B.3 · Tokenización (8 pts)
Qué debe entregar: **(4)** tabla de tokens, caracteres y tokens/carácter de los tres textos · **(1)** IDs y *decode* de un emoji · **(3)** explicación (4–5 líneas) de por qué el español consume más tokens y qué implica para costo y ventana de contexto.

In [ ]:
# [PROVISTO] Textos del enunciado y codificador.
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
TEXTOS = {
    "EN":     "Large language models predict the next token given the previous ones.",
    "ES":     "Los modelos de lenguaje grandes predicen el siguiente token a partir de los anteriores.",
    "CODIGO": "def f(x): return [i**2 for i in range(x) if i % 2 == 0]",
}

In [ ]:
# [SU TURNO] B.3 — tabla y emoji.

**[SU TURNO] B.3 — Explicación (3 pts).** *(escriba aquí)*

## SECCIÓN C — RAG: compatibilidad, recuperación, búsqueda híbrida e índices (32 pts)

### C.0 · Configuración y verificación de compatibilidad (6 pts)
Qué debe entregar: **(2)** `make_chat_client(provider)` para NVIDIA, Groq o HF (SDK `openai`, sin parámetros específicos de proveedor en la llamada común), proveedor y modelo declarados y un *ping* · **(2)** `embedding_info.json` leído; `assert` de dimensión y normalización; recálculo de **un** documento con el modelo oficial y coseno ≥ 0.99 contra el vector almacenado · **(2)** contraejemplo: otro modelo (misma dimensión, si es posible) o distinto `input_type`/prefijo, con su coseno y 2–3 líneas explicando R1 y R2.

In [ ]:
# [PROVISTO] URLs públicas de los proveedores (están en el enunciado). La función es SU TURNO.
from openai import OpenAI
PROVIDERS = {
    "nvidia": {"base_url": "https://integrate.api.nvidia.com/v1", "key": "NVIDIA_API_KEY"},
    "groq":   {"base_url": "https://api.groq.com/openai/v1",      "key": "GROQ_API_KEY"},
    "hf":     {"base_url": "https://router.huggingface.co/v1",    "key": "HF_TOKEN"},
}
PROVEEDOR   = "TODO"      # "nvidia" | "groq" | "hf"
MODELO_CHAT = "TODO"      # nombre exacto que funcionó en su ping (use client.models.list())

In [ ]:
# [SU TURNO] C.0.1
def make_chat_client(provider):
    # TODO: devuelva un OpenAI(base_url=..., api_key=...) leyendo la clave con get_secret()
    raise NotImplementedError("TODO")

# TODO: ping de una frase y muestre la respuesta. Cuidado con parámetros específicos de proveedor.

In [ ]:
# [PROVISTO] Lectura de datos de la sección C (el orden de Q coincide con preguntas_eval.json).
info    = json.load(open(DATA_DIR / "embedding_info.json"))
docs    = json.load(open(DATA_DIR / "corpus_reglamento.json"))
qs_all  = json.load(open(DATA_DIR / "preguntas_eval.json"))
E = np.load(DATA_DIR / "corpus_embeddings.npy"); Q = np.load(DATA_DIR / "question_embeddings.npy")
eval_idx = [i for i, q in enumerate(qs_all) if q["tipo"] in ("sigla", "parafrasis")]     # 12 preguntas
demo_idx = next(i for i, q in enumerate(qs_all) if q["tipo"] == "demo_fuera_corpus")    # fuera del corpus
print(info); print("E:", E.shape, "| Q:", Q.shape, "| preguntas de evaluación:", len(eval_idx))

In [ ]:
# [SU TURNO] C.0.2 y C.0.3 — assert de dimensión/normalización, recálculo de un documento con el modelo oficial
# (¡mismos parámetros que indica embedding_info.json!) y contraejemplo.

**[SU TURNO] C.0.3 — Explicación (R1 y R2).** *(escriba aquí)*

### C.1 · Búsqueda vectorial exacta (6 pts)
**(3)** `retrieve(q_vec, k)` con coseno por multiplicación matricial (sin FAISS) · **(3)** Recall@3 y MRR sobre las 12 preguntas de evaluación (`gold_doc_id`).

In [ ]:
# [SU TURNO] C.1
def retrieve(q_vec, k=3):
    # TODO: devuelva los índices de los k documentos más similares
    raise NotImplementedError("TODO")

### C.2 · BM25 y búsqueda híbrida (10 pts)
**(2)** BM25 con `rank_bm25` · **(2)** RRF: $\text{score}(d)=\sum_r 1/(60+\text{rank}_r(d))$ · **(3)** tabla de Recall@3 de los 3 métodos **separada por `tipo`** · **(3)** interpretación honesta (si algo contradice lo esperado, dígalo).

In [ ]:
# [SU TURNO] C.2
from rank_bm25 import BM25Okapi
# TODO: tokenización simple del corpus, BM25, rrf(rank_lists, k=60) y tabla por tipo.

**[SU TURNO] C.2 — Interpretación.** *(escriba aquí)*

### C.3 · Índices aproximados HNSW (5 pts)
Con `faiss-cpu`: **20 000 vectores aleatorios normalizados con la misma dimensión del corpus** y 100 consultas. **(3)** `IndexFlatIP` vs `IndexHNSWFlat(M=16)` para `efSearch` ∈ {8, 64}: recall@10 (el exacto es la verdad), latencia media por consulta y tiempo de construcción · **(2)** *trade-off* con sus números (¿nota que los vectores aleatorios son un caso difícil?).

In [ ]:
# [SU TURNO] C.3 — use rng = np.random.default_rng(SEED) y la dimensión E.shape[1]
import faiss

**[SU TURNO] C.3 — Trade-off.** *(escriba aquí)*

### C.4 · RAG de punta a punta con el LLM de su elección (5 pts)
Preguntas: `demo_fuera_corpus` + una `sigla` + una `parafrasis`. Contexto = los **3 chunks** recuperados con **su** `retrieve`. **(3)** por respuesta: afirmaciones, cuáles están respaldadas, **Faithfulness** · **(2)** análisis del caso fuera del corpus (¿se abstuvo o alucinó? ¿qué parte del prompt influyó?).
> Sin acceso a ningún proveedor: use `faithfulness_cases.json` (el ítem vale máx. 3 de 5).

In [ ]:
# [PROVISTO] Plantilla de prompt (la del laboratorio de RAG). Puede mejorarla: debe exigir responder SOLO con el contexto.
PROMPT = '''Eres un asistente institucional. Responde ÚNICAMENTE con el contexto proporcionado.
Si la información no está en el contexto, di explícitamente que no la tienes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:'''

In [ ]:
# [SU TURNO] C.4 — recupere, arme el prompt, genere con su cliente de C.0 e imprima pregunta, contexto y respuesta.

**[SU TURNO] C.4 — Tabla de fidelidad.** Una tabla por pregunta:

| # | Afirmación de la respuesta | ¿Respaldada por el contexto? (sí/no) |
|---|---|---|
| 1 | | |

**Faithfulness** = afirmaciones respaldadas / afirmaciones totales = *(calcule)*

**Análisis del caso fuera del corpus:** *(escriba aquí)*

## SECCIÓN D — Declaración de uso de IA y reproducibilidad (5 pts)

**[SU TURNO]** Complete todo:

1. **Herramienta(s) de IA usadas** y **2 prompts clave** (cópielos): …
2. **Proveedor, modelo de chat y modelo de embeddings efectivamente usados:** …
3. **Un caso concreto en que la IA produjo algo incorrecto o inadecuado** y cómo lo detectó y corrigió (2 pts): …
4. **Reproducibilidad:** semilla = …; versiones de librerías clave = … (por ejemplo `pip list | grep -i -E "torch|faiss|sentence|openai"`); el notebook corre de principio a fin: sí / no.

In [ ]:
# [PROVISTO] Versiones (pegue la salida en la sección D).
import importlib.metadata as md_
for p in ["torch", "transformers", "sentence-transformers", "faiss-cpu", "openai", "rank-bm25", "tiktoken"]:
    try: print(p, md_.version(p))
    except Exception: print(p, "no instalado")

## ✅ Revisión final antes de entregar

- [ ] **Reinicie y ejecute todo** (Entorno de ejecución → *Reiniciar y ejecutar todo*): debe correr de arriba hacia abajo **sin errores**.
- [ ] Todas las celdas `# [SU TURNO]` tienen su código/análisis, y **no quedan** `raise NotImplementedError` sin reemplazar.
- [ ] Las **salidas están visibles** (gráficas, tablas, asserts).
- [ ] **Ninguna celda ni salida muestra una API key** (ejecute la celda siguiente).
- [ ] Archivo guardado como `Eval2_Apellido_Nombre.ipynb`.
- [ ] Correo a **jipadillab@eafit.edu.co**, asunto = el que imprimió arriba, **antes de las 11:00 a. m.**

In [ ]:
# [PROVISTO] Buscador de claves. Guarde el notebook (Archivo → Guardar) y pase la ruta del .ipynb.
# En Colab, p. ej.: "/content/drive/MyDrive/Colab Notebooks/Eval2_Apellido_Nombre.ipynb"
import re, json
PATRONES = [r"nvapi-[A-Za-z0-9_\-]{10,}", r"gsk_[A-Za-z0-9]{10,}", r"hf_[A-Za-z0-9]{10,}"]
def buscar_keys(ruta_ipynb):
    nb = json.load(open(ruta_ipynb, encoding="utf-8"))
    valores = [v for v in (get_secret(k) for k in ["NVIDIA_API_KEY", "GROQ_API_KEY", "HF_TOKEN"]) if v]
    hallazgos = []
    for i, c in enumerate(nb["cells"]):
        txt = "".join(c.get("source", [])) + "".join(json.dumps(o) for o in c.get("outputs", []))
        if any(re.search(p, txt) for p in PATRONES) or any(v in txt for v in valores):
            hallazgos.append(i)
    if hallazgos:
        print("⚠️ Posible clave en las celdas:", hallazgos, "— bórrela de código Y de la salida, y regenere la clave.")
    else:
        print("✅ No se encontraron claves.")
# buscar_keys("RUTA/AL/NOTEBOOK.ipynb")
tiempo()